# Review Harmonization Suggestions

**Step 3 of 3** in the BioData Catalyst harmonization workflow:

1. [Download Studies Metadata](./download_studies_data_dictionaries.ipynb) — fetch preharmonized
   PFBs from BDC and extract the dbGaP data dictionary and variable report XML files.
2. [Generate Preliminary Harmonization Mappings](./harmonize_studies.ipynb) — map each study
   variable to ranked candidate slots in the target schema.
3. **Review Harmonization Suggestions** (this notebook) — accept or skip candidates one
   variable at a time.

| File | Content |
|------|---------|
| **Input** `{study_id}_preliminary_mappings.csv` | Top-N candidates per variable, from step 2 |
| **State** `{study_id}_review_state.csv` | Auto-written after every click; audit trail for resuming |
| **Output** `{study_id}_curated_mappings.csv` | Accepted variables — one row per variable, full mapping columns |
| **Output** `{study_id}_skipped_variables.csv` | Skipped variables — source info, best automated suggestion, empty `manual_mapping` column |

All three files live in `outputs/` and are rewritten on every accept or skip, so you can stop at
any point and pick up where you left off by simply re-running the notebook.

# Setup

## Libraries

Install with `pip install ai-harmonization ipywidgets`.

In [ ]:
# Uncomment to install
#!pip -q install ai-harmonization ipywidgets

In [ ]:
import os

from ai_harmonization.review import VariableReviewSession

## Configuration

Set the study to review. It must be one that step 2 produced a mapping file for; the default is
the same example study used by the first two notebooks.

In [ ]:
outputs_dir = './outputs'
study_id = 'phs000704.v1.p1.c1'

mapping_file = os.path.join(outputs_dir, f'{study_id}_preliminary_mappings.csv')
state_file = os.path.join(outputs_dir, f'{study_id}_review_state.csv')

if os.path.exists(mapping_file):
    print(f'Reviewing {mapping_file}')
else:
    print(f'{mapping_file} not found. Run harmonize_studies.ipynb for this study first.')

# Review

Use the buttons to navigate and record your choices (identical controls at top and bottom):

**Navigation** (no state change):
- **← Back** — go to the previous variable
- **→** — go to the next variable

**Decisions:**
- **Skip →** — mark as having no good mapping and advance (orange). The button turns red and
  reads **⊘ Skipped** when the variable on screen is already skipped; clicking it then clears
  the decision and stays put, so a mistaken skip can be undone by navigating back to it.
- **✓ #1 … #N** — accept the rank-N candidate and advance (green). One button is rendered per
  candidate rank present in the mapping file, so raising `top_n_suggestions` does not leave
  candidates unreachable. Ranks the current variable has no candidate for are greyed out.

Similarity color coding: **green ≥ 0.75** · **yellow 0.5–0.75** · **red < 0.5**

Progress bar shows `· auto-saving` when `auto_save` is active — every accept/skip writes all three output files automatically.

In [ ]:
# Works for both a first run and a resume:
#   - First run:  state_file not found → starts fresh, auto-save creates it on the first click
#   - Later runs: state_file found → restores accepted/skipped, jumps to first unreviewed variable
#
# sort_by options:
#   - 'similarity' (recommended) — variables with the highest best-match score shown first
#   - 'index'                    — variables in CSV order (original study order)
session = VariableReviewSession.from_csv(
    mapping_file,
    sort_by='similarity',
    resume_from=state_file,
    auto_save=state_file,
)
print(f'Loaded {session.n_variables} variables')

In [ ]:
# Run this cell once to launch the UI.
session.start()

# Session Summary

Re-run this cell at any point to see where the review stands and which files hold the
results.

In [ ]:
reviewed = len(session.curated_df) + len(session.skipped_df)
print(f'{session.n_variables} variables total')
print(f'  {len(session.curated_df)} accepted')
print(f'  {len(session.skipped_df)} skipped')
print(f'  {session.n_variables - reviewed} still to review')

# Written automatically on every accept/skip, but save() is safe to call again.
session.save(state_file)